# 04 Morgan、MACCS与传统机器学习基线

本Notebook仅使用20 ppm有效Development化合物，完成化合物级嵌套结构簇交叉验证。ChemVista外测标签和外测性能保持锁定。

表示：Morgan ECFP4（2048 bit）和RDKit MACCS（完整167 bit）。模型：均值/类别概率、标签打乱、正则化线性模型、随机森林、CatBoost、SVC/SVR。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from catboost import CatBoostClassifier, CatBoostRegressor, __version__ as catboost_version
from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss, confusion_matrix,
    matthews_corrcoef, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

SEED = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 4
N_BOOTSTRAP = 1000
N_LABEL_PERMUTATIONS = 50

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / 'outputs' / 'pbt_msms_v1').exists()), ROOT)
SPLIT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'split_v1'
AUDIT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'spectrum_audit_v1'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'structure_baselines_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'development': SPLIT_DIR / 'development_compounds.csv',
    'split_freeze': SPLIT_DIR / 'split_freeze_manifest.json',
    'split_qc': SPLIT_DIR / 'leakage_checks.csv',
    'development_applicability': SPLIT_DIR / 'development_cv_applicability.csv',
    'coverage': AUDIT_DIR / 'compound_spectrum_coverage.csv',
    'spectrum_audit': AUDIT_DIR / 'spectrum_audit_manifest.json',
    'spectrum_qc': AUDIT_DIR / 'spectrum_audit_qc_checks.csv',
}
assert all(path.exists() for path in PATHS.values()), [str(path) for path in PATHS.values() if not path.exists()]
print('Python:', sys.executable)
print('scikit-learn:', sklearn_version, '| CatBoost:', catboost_version, '| RDKit:', rdBase.rdkitVersion)
print('输出目录:', OUTPUT_DIR)

## 1. 冻结Development与20 ppm有效化合物

In [ ]:
development = pd.read_csv(PATHS['development'])
coverage = pd.read_csv(PATHS['coverage'])
development_applicability = pd.read_csv(PATHS['development_applicability'])
split_qc = pd.read_csv(PATHS['split_qc'])
spectrum_qc = pd.read_csv(PATHS['spectrum_qc'])
with PATHS['split_freeze'].open('r', encoding='utf-8') as handle:
    split_freeze = json.load(handle)
with PATHS['spectrum_audit'].open('r', encoding='utf-8') as handle:
    spectrum_audit = json.load(handle)

coverage_col = 'n_spectra__primary_train_eligible_20ppm'
coverage_map = coverage.set_index('inchikey_2d')[coverage_col]
development['n_eligible_spectra_20ppm'] = development['inchikey_2d'].map(coverage_map).fillna(0).astype(int)
for ppm in [50, 100]:
    ppm_col = f'n_spectra__primary_train_eligible_{ppm}ppm'
    development[f'n_eligible_spectra_{ppm}ppm'] = development['inchikey_2d'].map(coverage.set_index('inchikey_2d')[ppm_col]).fillna(0).astype(int)
modeling = development.loc[development['n_eligible_spectra_20ppm'].gt(0)].copy()
modeling['cv_fold'] = modeling['cv_fold'].astype(int)
modeling = modeling.sort_values('inchikey_2d').reset_index(drop=True)
modeling = modeling.merge(development_applicability, on=['inchikey_2d', 'cv_fold'], how='left', validate='one_to_one')
assert modeling['inchikey_2d'].is_unique
assert set(modeling['cv_fold']) == set(range(N_OUTER_FOLDS))
assert modeling.groupby('cv_group')['cv_fold'].nunique().max() == 1

TASKS = {
    'P_screen_not_ready_binary': {'column': 'label__P_screen_not_ready_binary', 'kind': 'classification'},
    'B_LOGBCF_REGRESSION': {'column': 'label__B_LOGBCF_REGRESSION', 'kind': 'regression'},
    'T_EC50_MG_L_REGRESSION': {'column': 'label__T_EC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_LC50_MG_L_REGRESSION': {'column': 'label__T_LC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_NOEC_MG_L_REGRESSION': {'column': 'label__T_NOEC_MG_L_REGRESSION', 'kind': 'regression'},
}
task_counts = []
for task, spec in TASKS.items():
    values = pd.to_numeric(modeling[spec['column']], errors='coerce')
    row = {'task': task, 'kind': spec['kind'], 'n_compounds': int(values.notna().sum())}
    if spec['kind'] == 'classification':
        row.update({'positive': int(values.eq(1).sum()), 'negative': int(values.eq(0).sum()), 'positive_rate': float(values.mean())})
    task_counts.append(row)
task_counts = pd.DataFrame(task_counts)
derived_threshold_rows = []
bcf_values = pd.to_numeric(modeling['label__B_LOGBCF_REGRESSION'], errors='coerce').dropna()
for name, threshold in [('B', np.log10(2000.0)), ('vB', np.log10(5000.0))]:
    derived_threshold_rows.append({'derived_class': name, 'n_compounds': len(bcf_values), 'positive': int(bcf_values.ge(threshold).sum()), 'negative': int(bcf_values.lt(threshold).sum()), 'positive_rate': float(bcf_values.ge(threshold).mean())})
for endpoint in ['EC50', 'LC50', 'NOEC']:
    values = pd.to_numeric(modeling[f'label__T_{endpoint}_MG_L_REGRESSION'], errors='coerce').dropna()
    derived_threshold_rows.append({'derived_class': f'{endpoint}_le_0.01mg_L', 'n_compounds': len(values), 'positive': int(values.le(-2.0).sum()), 'negative': int(values.gt(-2.0).sum()), 'positive_rate': float(values.le(-2.0).mean())})
derived_threshold_class_balance = pd.DataFrame(derived_threshold_rows)
display(task_counts)
display(derived_threshold_class_balance)
print('20 ppm有效Development:', len(modeling))

## 2. 生成Morgan与MACCS指纹

In [ ]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
morgan = np.zeros((len(modeling), 2048), dtype=np.uint8)
maccs = np.zeros((len(modeling), 167), dtype=np.uint8)
invalid_smiles = []
for i, row in tqdm(modeling.iterrows(), total=len(modeling), desc='计算结构指纹'):
    mol = Chem.MolFromSmiles(str(row['canonical_smiles']))
    if mol is None:
        invalid_smiles.append(row['inchikey_2d'])
        continue
    fp_morgan = morgan_generator.GetFingerprint(mol)
    fp_maccs = MACCSkeys.GenMACCSKeys(mol)
    DataStructs.ConvertToNumpyArray(fp_morgan, morgan[i])
    DataStructs.ConvertToNumpyArray(fp_maccs, maccs[i])
assert not invalid_smiles, invalid_smiles[:20]
assert morgan.shape == (len(modeling), 2048)
assert maccs.shape == (len(modeling), 167)
assert (morgan[:, 0] >= 0).all() and (maccs[:, 0] == 0).all()
np.savez_compressed(OUTPUT_DIR / 'structure_fingerprints.npz', morgan=morgan, maccs=maccs)
modeling[['inchikey_2d', 'canonical_smiles', 'cv_group', 'cv_fold', 'n_eligible_spectra_20ppm']].to_csv(OUTPUT_DIR / 'structure_fingerprint_row_index.csv', index=False)
print('Morgan:', morgan.shape, '| MACCS:', maccs.shape)

## 3. 模型、嵌套CV与评价函数

In [ ]:
def classification_model_specs(seed):
    return {
        'logistic': (LogisticRegression(max_iter=4000, solver='liblinear', random_state=seed), {'C': [0.03, 0.3, 3.0, 30.0]}),
        'random_forest': (RandomForestClassifier(n_estimators=400, random_state=seed, n_jobs=-1), {'max_depth': [None, 10], 'min_samples_leaf': [1, 3]}),
        'catboost': (CatBoostClassifier(iterations=300, loss_function='Logloss', verbose=False, allow_writing_files=False, random_seed=seed, thread_count=-1), {'depth': [4, 6], 'learning_rate': [0.03, 0.1]}),
        'svc_rbf': (SVC(kernel='rbf'), {'C': [0.3, 3.0, 30.0], 'gamma': ['scale', 0.01]}),
    }

def regression_model_specs(seed):
    return {
        'ridge': (Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}),
        'random_forest': (RandomForestRegressor(n_estimators=400, random_state=seed, n_jobs=-1), {'max_depth': [None, 10], 'min_samples_leaf': [1, 3]}),
        'catboost': (CatBoostRegressor(iterations=400, loss_function='RMSE', verbose=False, allow_writing_files=False, random_seed=seed, thread_count=-1), {'depth': [4, 6], 'learning_rate': [0.03, 0.1]}),
        'svr_rbf': (TransformedTargetRegressor(regressor=SVR(kernel='rbf'), transformer=StandardScaler()), {'regressor__C': [0.3, 3.0, 30.0], 'regressor__gamma': ['scale', 0.01], 'regressor__epsilon': [0.05, 0.2]}),
    }

def classification_metrics(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'pr_auc': average_precision_score(y_true, probability),
        'roc_auc': roc_auc_score(y_true, probability),
        'mcc': matthews_corrcoef(y_true, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_true, prediction),
        'sensitivity': tp / (tp + fn) if tp + fn else np.nan,
        'specificity': tn / (tn + fp) if tn + fp else np.nan,
        'brier': brier_score_loss(y_true, probability),
    }

def regression_metrics(y_true, prediction):
    y_true = np.asarray(y_true, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    rho = spearmanr(y_true, prediction).statistic if len(y_true) > 1 else np.nan
    return {
        'mae': mean_absolute_error(y_true, prediction),
        'rmse': mean_squared_error(y_true, prediction) ** 0.5,
        'r2': r2_score(y_true, prediction),
        'spearman': rho,
    }

def json_params(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)

## 4. 运行五个主端点的嵌套结构簇CV

In [ ]:
REPRESENTATIONS = {'Morgan_2048': morgan, 'MACCS_167': maccs}
prediction_rows = []
search_rows = []
total_jobs = len(REPRESENTATIONS) * len(TASKS) * N_OUTER_FOLDS
progress = tqdm(total=total_jobs, desc='嵌套CV：表示×任务×外层折', mininterval=2.0)

for representation, X_all in REPRESENTATIONS.items():
    for task, task_spec in TASKS.items():
        label_values = pd.to_numeric(modeling[task_spec['column']], errors='coerce')
        valid_indices = np.flatnonzero(label_values.notna().to_numpy())
        X = X_all[valid_indices]
        y = label_values.iloc[valid_indices].to_numpy(dtype=float)
        keys = modeling.iloc[valid_indices]['inchikey_2d'].to_numpy()
        groups = modeling.iloc[valid_indices]['cv_group'].astype(str).to_numpy()
        outer_folds = modeling.iloc[valid_indices]['cv_fold'].to_numpy(dtype=int)
        for outer_fold in range(N_OUTER_FOLDS):
            test_mask = outer_folds == outer_fold
            train_mask = ~test_mask
            X_train, X_test = X[train_mask], X[test_mask]
            y_train, y_test = y[train_mask], y[test_mask]
            groups_train = groups[train_mask]
            keys_test = keys[test_mask]
            fold_seed = SEED + outer_fold

            if task_spec['kind'] == 'classification':
                y_train_int = y_train.astype(int)
                y_test_int = y_test.astype(int)
                inner_splitter = StratifiedGroupKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=fold_seed)
                inner_splits = list(inner_splitter.split(X_train, y_train_int, groups_train))
                prevalence = float(y_train_int.mean())
                dummy_probability = np.full(len(y_test_int), prevalence)
                for key, truth, probability in zip(keys_test, y_test_int, dummy_probability):
                    prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'classification', 'model': 'dummy_prevalence', 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': float(probability >= 0.5), 'y_probability': probability, 'inner_best_score': np.nan, 'best_params': '{}'})

                for permutation_index in range(N_LABEL_PERMUTATIONS):
                    rng = np.random.default_rng(fold_seed + 10_000 + permutation_index * 1_000)
                    shuffled = rng.permutation(y_train_int)
                    permutation_model = LogisticRegression(C=1.0, max_iter=4000, solver='liblinear', random_state=fold_seed).fit(X_train, shuffled)
                    permutation_probability = permutation_model.predict_proba(X_test)[:, 1]
                    permutation_name = f'label_permutation_{permutation_index:03d}'
                    for key, truth, probability in zip(keys_test, y_test_int, permutation_probability):
                        prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'classification', 'model': permutation_name, 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': float(probability >= 0.5), 'y_probability': probability, 'inner_best_score': np.nan, 'best_params': '{}'})

                for model_name, (estimator, grid) in classification_model_specs(fold_seed).items():
                    search = GridSearchCV(estimator, grid, scoring='average_precision', cv=inner_splits, refit=True, n_jobs=1, error_score='raise')
                    search.fit(X_train, y_train_int)
                    calibrated = CalibratedClassifierCV(estimator=clone(search.best_estimator_), method='sigmoid', cv=inner_splits, ensemble=False, n_jobs=1)
                    calibrated.fit(X_train, y_train_int)
                    probability = calibrated.predict_proba(X_test)[:, 1]
                    params = json_params(search.best_params_)
                    search_rows.append({'representation': representation, 'task': task, 'model': model_name, 'outer_fold': outer_fold, 'inner_best_score': float(search.best_score_), 'best_params': params})
                    for key, truth, prob in zip(keys_test, y_test_int, probability):
                        prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'classification', 'model': model_name, 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': float(prob >= 0.5), 'y_probability': prob, 'inner_best_score': float(search.best_score_), 'best_params': params})
            else:
                inner_splitter = GroupKFold(n_splits=N_INNER_FOLDS)
                inner_splits = list(inner_splitter.split(X_train, y_train, groups_train))
                dummy_prediction = np.full(len(y_test), float(y_train.mean()))
                for key, truth, pred in zip(keys_test, y_test, dummy_prediction):
                    prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'regression', 'model': 'dummy_mean', 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': pred, 'y_probability': np.nan, 'inner_best_score': np.nan, 'best_params': '{}'})

                for permutation_index in range(N_LABEL_PERMUTATIONS):
                    rng = np.random.default_rng(fold_seed + 20_000 + permutation_index * 1_000)
                    shuffled = rng.permutation(y_train)
                    permutation_model = Ridge(alpha=1.0).fit(X_train, shuffled)
                    permutation_prediction = permutation_model.predict(X_test)
                    permutation_name = f'label_permutation_{permutation_index:03d}'
                    for key, truth, pred in zip(keys_test, y_test, permutation_prediction):
                        prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'regression', 'model': permutation_name, 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': pred, 'y_probability': np.nan, 'inner_best_score': np.nan, 'best_params': '{}'})

                for model_name, (estimator, grid) in regression_model_specs(fold_seed).items():
                    search = GridSearchCV(estimator, grid, scoring='neg_mean_absolute_error', cv=inner_splits, refit=True, n_jobs=1, error_score='raise')
                    search.fit(X_train, y_train)
                    prediction = search.best_estimator_.predict(X_test)
                    params = json_params(search.best_params_)
                    search_rows.append({'representation': representation, 'task': task, 'model': model_name, 'outer_fold': outer_fold, 'inner_best_score': float(search.best_score_), 'best_params': params})
                    for key, truth, pred in zip(keys_test, y_test, prediction):
                        prediction_rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': 'regression', 'model': model_name, 'outer_fold': outer_fold, 'y_true': truth, 'y_prediction': float(pred), 'y_probability': np.nan, 'inner_best_score': float(search.best_score_), 'best_params': params})
            progress.update(1)
progress.close()
oof = pd.DataFrame(prediction_rows)
search_results = pd.DataFrame(search_rows)
nested_selected_rows = []
selected_family_rows = []
for (representation, task, outer_fold), frame in search_results.groupby(['representation', 'task', 'outer_fold'], sort=True):
    winner = frame.sort_values(['inner_best_score', 'model'], ascending=[False, True]).iloc[0]
    selected = oof.loc[
        oof['representation'].eq(representation) & oof['task'].eq(task)
        & oof['outer_fold'].eq(outer_fold) & oof['model'].eq(winner['model'])
    ].copy()
    selected['selected_family'] = winner['model']
    selected['model'] = 'nested_selected'
    nested_selected_rows.append(selected)
    selected_family_rows.append({'representation': representation, 'task': task, 'outer_fold': int(outer_fold), 'selected_family': winner['model'], 'inner_best_score': float(winner['inner_best_score']), 'best_params': winner['best_params']})
oof = pd.concat([oof, *nested_selected_rows], ignore_index=True, sort=False)
selected_families = pd.DataFrame(selected_family_rows)
print('OOF预测行数:', len(oof), '| 调参记录:', len(search_results))

## 5. OOF指标、内层模型家族选择、置换检验与敏感性分析

In [ ]:
metric_rows = []
fold_metric_rows = []
bootstrap_rng = np.random.default_rng(SEED + 99_000)
for (representation, task, model), frame in oof.groupby(['representation', 'task', 'model'], sort=True):
    kind = frame['kind'].iloc[0]
    is_permutation = str(model).startswith('label_permutation_')
    if kind == 'classification':
        metrics = classification_metrics(frame['y_true'], frame['y_probability'])
        primary_name = 'pr_auc'
        primary_values = []
        y_true = frame['y_true'].to_numpy(dtype=int)
        y_score = frame['y_probability'].to_numpy(dtype=float)
        for _ in range(0 if is_permutation else N_BOOTSTRAP):
            idx = bootstrap_rng.integers(0, len(frame), len(frame))
            if np.unique(y_true[idx]).size == 2:
                primary_values.append(average_precision_score(y_true[idx], y_score[idx]))
    else:
        metrics = regression_metrics(frame['y_true'], frame['y_prediction'])
        primary_name = 'mae'
        primary_values = []
        y_true = frame['y_true'].to_numpy(dtype=float)
        y_pred = frame['y_prediction'].to_numpy(dtype=float)
        for _ in range(0 if is_permutation else N_BOOTSTRAP):
            idx = bootstrap_rng.integers(0, len(frame), len(frame))
            primary_values.append(mean_absolute_error(y_true[idx], y_pred[idx]))
    row = {'representation': representation, 'task': task, 'kind': kind, 'model': model, 'n_compounds': len(frame), **metrics}
    row['primary_metric'] = primary_name
    row['primary_value'] = metrics[primary_name]
    row['primary_ci_low'] = float(np.quantile(primary_values, 0.025)) if primary_values else np.nan
    row['primary_ci_high'] = float(np.quantile(primary_values, 0.975)) if primary_values else np.nan
    metric_rows.append(row)
    for outer_fold, fold_frame in frame.groupby('outer_fold'):
        fold_metrics = classification_metrics(fold_frame['y_true'], fold_frame['y_probability']) if kind == 'classification' else regression_metrics(fold_frame['y_true'], fold_frame['y_prediction'])
        fold_metric_rows.append({'representation': representation, 'task': task, 'kind': kind, 'model': model, 'outer_fold': int(outer_fold), 'n_compounds': len(fold_frame), **fold_metrics})
metrics = pd.DataFrame(metric_rows)
fold_metrics = pd.DataFrame(fold_metric_rows)
metrics['candidate_model'] = ~metrics['model'].str.startswith(('dummy_', 'label_permutation_'))
metrics['rank'] = np.nan
for (representation, task), idx in metrics.loc[metrics['candidate_model']].groupby(['representation', 'task']).groups.items():
    ascending = metrics.loc[idx, 'kind'].iloc[0] == 'regression'
    metrics.loc[idx, 'rank'] = metrics.loc[idx, 'primary_value'].rank(method='min', ascending=ascending)
metrics['rank'] = metrics['rank'].astype('Int64')
best_models = metrics.loc[metrics['model'].eq('nested_selected')].sort_values(['task', 'representation'])
selected_family_frequency = selected_families.groupby(['representation', 'task', 'selected_family']).size().rename('n_outer_folds_selected').reset_index()
permutation_metric_rows = metrics.loc[metrics['model'].str.startswith('label_permutation_')].copy()
permutation_summary_rows = []
for (representation, task), null_frame in permutation_metric_rows.groupby(['representation', 'task']):
    observed = best_models.loc[best_models['representation'].eq(representation) & best_models['task'].eq(task)].iloc[0]
    null_values = null_frame['primary_value'].to_numpy(dtype=float)
    if observed['kind'] == 'classification':
        empirical_p = (1 + int((null_values >= observed['primary_value']).sum())) / (1 + len(null_values))
    else:
        empirical_p = (1 + int((null_values <= observed['primary_value']).sum())) / (1 + len(null_values))
    permutation_summary_rows.append({'representation': representation, 'task': task, 'kind': observed['kind'], 'n_permutations': len(null_values), 'observed_nested_primary': observed['primary_value'], 'null_mean': float(np.mean(null_values)), 'null_q05': float(np.quantile(null_values, 0.05)), 'null_q95': float(np.quantile(null_values, 0.95)), 'empirical_p': empirical_p})
permutation_summary = pd.DataFrame(permutation_summary_rows)
display(best_models[['representation', 'task', 'model', 'n_compounds', 'primary_metric', 'primary_value', 'primary_ci_low', 'primary_ci_high']])

def fixed_linear_oof(X_all, frame, target_column, kind, representation, task, analysis_name, random_split=False):
    values = pd.to_numeric(frame[target_column], errors='coerce')
    valid_indices = np.flatnonzero(values.notna().to_numpy())
    X = X_all[valid_indices]
    y = values.iloc[valid_indices].to_numpy(dtype=float)
    keys = frame.iloc[valid_indices]['inchikey_2d'].to_numpy()
    if random_split:
        splitter = StratifiedKFold(N_OUTER_FOLDS, shuffle=True, random_state=SEED) if kind == 'classification' else KFold(N_OUTER_FOLDS, shuffle=True, random_state=SEED)
        split_iterator = splitter.split(X, y.astype(int) if kind == 'classification' else None)
    else:
        folds = frame.iloc[valid_indices]['cv_fold'].to_numpy(dtype=int)
        split_iterator = ((np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)) for fold in range(N_OUTER_FOLDS))
    rows = []
    for fold, (train_index, test_index) in enumerate(split_iterator):
        if kind == 'classification':
            estimator = LogisticRegression(C=1.0, max_iter=4000, solver='liblinear', random_state=SEED + fold)
            estimator.fit(X[train_index], y[train_index].astype(int))
            probability = estimator.predict_proba(X[test_index])[:, 1]
            prediction = (probability >= 0.5).astype(float)
        else:
            estimator = Ridge(alpha=10.0)
            estimator.fit(X[train_index], y[train_index])
            prediction = estimator.predict(X[test_index])
            probability = np.full(len(test_index), np.nan)
        for key, truth, pred, prob in zip(keys[test_index], y[test_index], prediction, probability):
            rows.append({'inchikey_2d': key, 'representation': representation, 'task': task, 'kind': kind, 'analysis': analysis_name, 'outer_fold': fold, 'y_true': truth, 'y_prediction': float(pred), 'y_probability': float(prob) if np.isfinite(prob) else np.nan})
    return pd.DataFrame(rows)

def summarize_fixed_oof(frame):
    rows = []
    for keys, group in frame.groupby(['representation', 'task', 'kind', 'analysis'], sort=True):
        representation, task, kind, analysis = keys
        result = classification_metrics(group['y_true'], group['y_probability']) if kind == 'classification' else regression_metrics(group['y_true'], group['y_prediction'])
        rows.append({'representation': representation, 'task': task, 'kind': kind, 'analysis': analysis, 'n_compounds': len(group), **result})
    return pd.DataFrame(rows)

label_variant_columns = {}
for task, spec in TASKS.items():
    label_variant_columns[(task, 'primary_median')] = spec['column']
    label_variant_columns[(task, 'direct_experimental_median')] = spec['column'].replace('label__', 'label_experimental__')
    label_variant_columns[(task, 'high_quality_median')] = spec['column'].replace('label__', 'label_hq__')
    if task == 'B_LOGBCF_REGRESSION':
        label_variant_columns[(task, 'risk_conservative')] = spec['column'].replace('label__', 'max__')
    elif spec['kind'] == 'regression':
        label_variant_columns[(task, 'risk_conservative')] = spec['column'].replace('label__', 'min__')
    else:
        label_variant_columns[(task, 'risk_conservative')] = spec['column']
label_sensitivity_frames = []
for representation, X_all in REPRESENTATIONS.items():
    for (task, variant), column in label_variant_columns.items():
        if column in modeling.columns:
            label_sensitivity_frames.append(fixed_linear_oof(X_all, modeling, column, TASKS[task]['kind'], representation, task, variant))
label_sensitivity_oof = pd.concat(label_sensitivity_frames, ignore_index=True)
label_sensitivity_metrics = summarize_fixed_oof(label_sensitivity_oof)

random_split_frames = []
for representation, X_all in REPRESENTATIONS.items():
    for task, spec in TASKS.items():
        random_split_frames.append(fixed_linear_oof(X_all, modeling, spec['column'], spec['kind'], representation, task, 'random_compound_5fold', random_split=True))
        random_split_frames.append(label_sensitivity_oof.loc[label_sensitivity_oof['representation'].eq(representation) & label_sensitivity_oof['task'].eq(task) & label_sensitivity_oof['analysis'].eq('primary_median')].assign(analysis='frozen_structure_cluster_5fold'))
split_strategy_oof = pd.concat(random_split_frames, ignore_index=True)
split_strategy_metrics = summarize_fixed_oof(split_strategy_oof)

development_for_ppm = development.sort_values('inchikey_2d').reset_index(drop=True).copy()
development_morgan = np.zeros((len(development_for_ppm), 2048), dtype=np.uint8)
development_maccs = np.zeros((len(development_for_ppm), 167), dtype=np.uint8)
for i, smiles in enumerate(development_for_ppm['canonical_smiles']):
    mol = Chem.MolFromSmiles(str(smiles))
    DataStructs.ConvertToNumpyArray(morgan_generator.GetFingerprint(mol), development_morgan[i])
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), development_maccs[i])
ppm_frames = []
for ppm in [20, 50, 100]:
    eligible = development_for_ppm[f'n_eligible_spectra_{ppm}ppm'].gt(0).to_numpy()
    ppm_frame = development_for_ppm.loc[eligible].reset_index(drop=True)
    for representation, X_all in {'Morgan_2048': development_morgan[eligible], 'MACCS_167': development_maccs[eligible]}.items():
        for task, spec in TASKS.items():
            ppm_frames.append(fixed_linear_oof(X_all, ppm_frame, spec['column'], spec['kind'], representation, task, f'{ppm}ppm'))
ppm_sensitivity_oof = pd.concat(ppm_frames, ignore_index=True)
ppm_sensitivity_metrics = summarize_fixed_oof(ppm_sensitivity_oof)

nested_oof = oof.loc[oof['model'].eq('nested_selected')].merge(modeling[['inchikey_2d', 'max_morgan_tanimoto_to_outer_train']], on='inchikey_2d', how='left', validate='many_to_one')
nested_oof['ood_bin'] = pd.cut(nested_oof['max_morgan_tanimoto_to_outer_train'], bins=[-np.inf, 0.3, 0.5, 0.7, np.inf], labels=['le_0.3', '0.3_0.5', '0.5_0.7', 'gt_0.7'])
ood_rows = []
for (representation, task, kind, ood_bin), group in nested_oof.groupby(['representation', 'task', 'kind', 'ood_bin'], observed=True):
    if len(group) < 10 or (kind == 'classification' and group['y_true'].nunique() < 2):
        continue
    result = classification_metrics(group['y_true'], group['y_probability']) if kind == 'classification' else regression_metrics(group['y_true'], group['y_prediction'])
    ood_rows.append({'representation': representation, 'task': task, 'kind': kind, 'ood_bin': str(ood_bin), 'n_compounds': len(group), 'median_max_train_tanimoto': float(group['max_morgan_tanimoto_to_outer_train'].median()), **result})
ood_metrics = pd.DataFrame(ood_rows)
display(permutation_summary)
display(label_sensitivity_metrics)
display(split_strategy_metrics)

## 6. 验收、冻结输出与运行清单

In [ ]:
checks = []
def add_check(check_id, description, passed, details=''):
    checks.append({'check_id': check_id, 'description': description, 'status': 'PASS' if passed else 'FAIL', 'details': details})

add_check('BASE001', '上游拆分无FAIL', not split_qc['status'].eq('FAIL').any())
add_check('BASE002', '上游谱图审计无FAIL', not spectrum_qc['status'].eq('FAIL').any())
add_check('BASE003', '仅使用20ppm有效Development', len(modeling) == spectrum_audit['counts']['primary_train_20ppm_compounds'], f"{len(modeling)} / {spectrum_audit['counts']['primary_train_20ppm_compounds']}")
add_check('BASE004', 'Morgan维数为2048且MACCS维数为167', morgan.shape[1] == 2048 and maccs.shape[1] == 167, f'{morgan.shape[1]} / {maccs.shape[1]}')
add_check('BASE005', '同一cv_group只属于一个外层fold', modeling.groupby('cv_group')['cv_fold'].nunique().max() == 1)
expected_models = {'dummy_prevalence', 'logistic', 'random_forest', 'catboost', 'svc_rbf', 'dummy_mean', 'ridge', 'svr_rbf', 'nested_selected'}
add_check('BASE006', '所有计划模型均产生OOF预测', expected_models.issubset(set(oof['model'])), ','.join(sorted(set(oof['model']))))
oof_counts_ok = True
for representation in REPRESENTATIONS:
    for task, spec in TASKS.items():
        expected_n = int(pd.to_numeric(modeling[spec['column']], errors='coerce').notna().sum())
        subset = oof[(oof['representation'] == representation) & (oof['task'] == task)]
        if not subset.groupby('model')['inchikey_2d'].nunique().eq(expected_n).all():
            oof_counts_ok = False
add_check('BASE007', '每个模型均覆盖相应任务全部有效化合物一次', oof_counts_ok)
add_check('BASE008', 'OOF预测无缺失值', oof['y_prediction'].notna().all() and oof['y_true'].notna().all())
add_check('BASE009', '分类概率均位于0到1', oof.loc[oof['kind'].eq('classification'), 'y_probability'].between(0, 1).all())
add_check('BASE010', '每个候选模型均完成5个外层fold', search_results.groupby(['representation', 'task', 'model'])['outer_fold'].nunique().eq(N_OUTER_FOLDS).all())
add_check('BASE011', '每个任务和表示均有50次完整标签置换', metrics.loc[metrics['model'].str.startswith('label_permutation_')].groupby(['representation', 'task'])['model'].nunique().eq(N_LABEL_PERMUTATIONS).all())
add_check('BASE012', '模型家族选择完全在外层训练集的内层CV完成', selected_families.groupby(['representation', 'task'])['outer_fold'].nunique().eq(N_OUTER_FOLDS).all())
add_check('BASE013', '四套标签敏感性均产生结果', set(['primary_median', 'direct_experimental_median', 'high_quality_median', 'risk_conservative']).issubset(set(label_sensitivity_metrics['analysis'])))
add_check('BASE014', '随机与结构簇拆分消融均完成', set(split_strategy_metrics['analysis']) == {'random_compound_5fold', 'frozen_structure_cluster_5fold'})
add_check('BASE015', '20/50/100ppm敏感性均完成', set(ppm_sensitivity_metrics['analysis']) == {'20ppm', '50ppm', '100ppm'})
add_check('BASE016', 'OOD分层使用冻结外层训练相似度且无缺失', nested_oof['max_morgan_tanimoto_to_outer_train'].notna().all())
qc = pd.DataFrame(checks)
assert qc['status'].eq('PASS').all(), qc[qc['status'].eq('FAIL')]

oof.to_pickle(OUTPUT_DIR / 'structure_baseline_oof_predictions.pkl')
oof.to_csv(OUTPUT_DIR / 'structure_baseline_oof_predictions.csv.gz', index=False, compression='gzip')
search_results.to_csv(OUTPUT_DIR / 'nested_cv_hyperparameters.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'structure_baseline_metrics.csv', index=False)
fold_metrics.to_csv(OUTPUT_DIR / 'structure_baseline_fold_metrics.csv', index=False)
best_models.to_csv(OUTPUT_DIR / 'structure_baseline_best_models.csv', index=False)
selected_families.to_csv(OUTPUT_DIR / 'nested_selected_family_by_fold.csv', index=False)
selected_family_frequency.to_csv(OUTPUT_DIR / 'nested_selected_family_frequency.csv', index=False)
permutation_summary.to_csv(OUTPUT_DIR / 'label_permutation_summary.csv', index=False)
label_sensitivity_oof.to_csv(OUTPUT_DIR / 'structure_baseline_label_sensitivity_oof.csv.gz', index=False, compression='gzip')
label_sensitivity_metrics.to_csv(OUTPUT_DIR / 'structure_baseline_label_sensitivity_metrics.csv', index=False)
split_strategy_metrics.to_csv(OUTPUT_DIR / 'structure_baseline_split_strategy_sensitivity.csv', index=False)
ppm_sensitivity_metrics.to_csv(OUTPUT_DIR / 'structure_baseline_ppm_sensitivity.csv', index=False)
ood_metrics.to_csv(OUTPUT_DIR / 'structure_baseline_ood_metrics.csv', index=False)
derived_threshold_class_balance.to_csv(OUTPUT_DIR / 'derived_threshold_class_balance.csv', index=False)
task_counts.to_csv(OUTPUT_DIR / 'structure_baseline_task_counts.csv', index=False)
qc.to_csv(OUTPUT_DIR / 'structure_baseline_qc_checks.csv', index=False)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

row_hash = hashlib.sha256('\n'.join(modeling['inchikey_2d']).encode('utf-8')).hexdigest()
run_manifest = {
    'run_version': 'PBT_MSMS_STRUCTURE_BASELINES_V2',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'random_seed': SEED,
    'external_test_unlocked': False,
    'development_definition': '20ppm eligible Development compounds only',
    'n_development_compounds': len(modeling),
    'development_ik2d_sha256': row_hash,
    'representations': {
        'Morgan_2048': {'radius': 2, 'n_bits': 2048, 'use_chirality': False},
        'MACCS_167': {'implementation': 'RDKit MACCS Keys', 'n_bits': 167},
    },
    'outer_cv': {'method': 'frozen Morgan-Butina structure clusters', 'n_folds': N_OUTER_FOLDS},
    'inner_cv': {'classification': 'StratifiedGroupKFold', 'regression': 'GroupKFold', 'n_folds': N_INNER_FOLDS},
    'models': {'classification': list(classification_model_specs(SEED)), 'regression': list(regression_model_specs(SEED)), 'controls': ['dummy', f'{N_LABEL_PERMUTATIONS}_label_permutations'], 'confirmatory_oof': 'nested_selected_by_inner_cv'},
    'tasks': TASKS,
    'bootstrap_resamples': N_BOOTSTRAP,
    'bootstrap_scope': 'compound-level conditional OOF bootstrap; does not include model retraining variability',
    'label_sensitivity_layers': ['primary_median', 'direct_experimental_median', 'high_quality_median', 'risk_conservative'],
    'sensitivity_head': {'classification': 'fixed logistic C=1', 'regression': 'fixed ridge alpha=10', 'purpose': 'paired robustness analysis rather than performance maximization'},
    'additional_sensitivity': ['random_vs_structure_cluster_split', '20_50_100ppm_eligibility', 'max_train_tanimoto_OOD'],
    'input_hashes': {name: sha256_file(path) for name, path in PATHS.items()},
    'software': {'python': sys.version, 'executable': sys.executable, 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn_version, 'catboost': catboost_version, 'rdkit': rdBase.rdkitVersion, 'platform': platform.platform()},
    'qc_status': 'PASS',
}
with (OUTPUT_DIR / 'structure_baseline_run_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)
    handle.write('\n')
display(qc)
print('全部结构基线完成；ChemVista外测仍未解锁。')